# Входной аудит данных перед аналитическим отчётом

Практическая ситуация: вы получили несколько файлов с данными о продажах, товарах, клиентах и регионах. Руководитель просит подготовить отчёт по выручке, но перед расчётами нужно понять, можно ли доверять данным.

Главный результат notebook: сводка качества данных, список найденных проблем и решение аналитика о пригодности данных для отчёта.

## Что нужно получить в конце

После выполнения notebook в папке `outputs` должны появиться файлы:

- `data_quality_summary.csv` — сводка качества по каждому набору данных;
- `data_quality_issues.csv` — список проблем с уровнем критичности;
- `analyst_decision.md` — короткое решение аналитика.

## 1. Подготовка окружения

    Сначала проверим, из какой папки запущен notebook, где лежат исходные данные и куда будут сохраняться результаты.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

def find_project_root(start=None):
    """Находит корень учебного проекта.

    Функция нужна, чтобы notebook работал и при запуске из корня проекта,
    и при запуске из папки notebooks.
    """
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, start.parent]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError(
        "Не найдена папка data/raw. Откройте notebook из корня проекта "
        "или из папки notebooks внутри проекта."
    )

project_root = find_project_root()
data_dir = project_root / "data" / "raw"
output_dir = project_root / "outputs"
output_dir.mkdir(exist_ok=True)

print("Корень проекта:", project_root)
print("Папка с исходными данными:", data_dir)
print("Папка для результатов:", output_dir)

## 2. Проверка наличия файлов

    Этот блок нужен для диагностики. Если хотя бы один файл не найден, дальше анализировать данные нельзя.

In [ ]:
required_files = {
    "sales": data_dir / "sales.csv",
    "products": data_dir / "products.xlsx",
    "clients": data_dir / "clients.csv",
    "regions": data_dir / "regions.json",
}

print("Проверяем файлы:")
missing_files = []
for name, file_path in required_files.items():
    if file_path.exists():
        print("OK:", file_path.relative_to(project_root))
    else:
        print("НЕ НАЙДЕН:", file_path.relative_to(project_root))
        missing_files.append(file_path)

if missing_files:
    raise FileNotFoundError("Не найдены обязательные файлы. Проверьте структуру проекта.")

## 3. Загрузка данных

    Загружаем фактовую таблицу продаж и три справочника. После загрузки сразу смотрим количество строк и столбцов.

In [ ]:
sales = pd.read_csv(required_files["sales"])
products = pd.read_excel(required_files["products"], sheet_name="products")
clients = pd.read_csv(required_files["clients"])
regions = pd.read_json(required_files["regions"])

datasets = {
    "sales": sales,
    "products": products,
    "clients": clients,
    "regions": regions,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} строк, {df.shape[1]} столбцов")

## 4. Карта источников

    Перед проверками важно понять роль каждого файла. Фактовая таблица отвечает на вопрос «что произошло», а справочники помогают объяснить продажи через товары, клиентов и регионы.

In [ ]:
source_map = pd.DataFrame({
    "dataset": ["sales", "products", "clients", "regions"],
    "role": [
        "факты продаж",
        "справочник товаров",
        "справочник клиентов",
        "справочник регионов",
    ],
    "key_fields": [
        "order_id, product_id, client_id, region_id",
        "product_id",
        "client_id",
        "region_id",
    ],
    "business_risk": [
        "ошибки искажают расчет выручки и динамики",
        "ошибки искажают аналитику по товарам и категориям",
        "ошибки искажают аналитику по клиентским сегментам",
        "ошибки искажают сравнение регионов",
    ],
})

display(source_map)

## 5. Быстрый аудит структуры

    На этом этапе мы ещё не исправляем данные. Мы собираем первую диагностическую картину: размеры таблиц, пропуски и полные дубли.

In [ ]:
summary_rows = []
for name, df in datasets.items():
    summary_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicated_rows": int(df.duplicated().sum()),
    })

data_quality_summary = pd.DataFrame(summary_rows)
display(data_quality_summary)

## 6. Проверка типов данных

    Типы данных важны для расчётов. Даты, числа и текст обрабатываются по-разному.

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    dtype_table = df.dtypes.astype(str).reset_index()
    dtype_table.columns = ["column", "dtype"]
    display(dtype_table)

## 7. Проверка дат

    Дату заказа нужно преобразовать в тип даты. Значения, которые не удалось распознать, станут `NaT`. Такие строки нужно проверить отдельно.

In [ ]:
sales["order_date_parsed"] = pd.to_datetime(sales["order_date"], errors="coerce")
invalid_dates = sales[sales["order_date_parsed"].isna()].copy()

print("Количество нераспознанных дат:", len(invalid_dates))
display(invalid_dates[["order_id", "order_date", "client_id", "product_id", "region_id"]].head(10))

**Контрольный вопрос:** можно ли строить динамику продаж, если часть дат не распознана? Запишите ответ своими словами в итоговом файле `analyst_decision.md`.

## 8. Проверка дублей заказов

    Для отчёта опасны не только полностью одинаковые строки. Особенно важно проверить бизнес-ключ `order_id`: если один заказ попал в выгрузку несколько раз, выручка может быть завышена.

In [ ]:
duplicated_orders = sales[sales["order_id"].duplicated(keep=False)].sort_values("order_id")

print("Полных дублей строк в sales:", int(sales.duplicated().sum()))
print("Строк с повторяющимися order_id:", len(duplicated_orders))
display(duplicated_orders.head(12))

## 9. Проверка бизнес-правил

    Проверяем невозможные или подозрительные значения: отрицательное количество, нулевую цену, скидки вне диапазона, нераспознанные даты и пустые регионы.

In [ ]:
checks = {
    "quantity <= 0": sales["quantity"] <= 0,
    "unit_price <= 0": sales["unit_price"] <= 0,
    "discount < 0 or discount > 1": (sales["discount"] < 0) | (sales["discount"] > 1),
    "invalid order_date": sales["order_date_parsed"].isna(),
    "missing region_id": sales["region_id"].isna(),
}

business_rules_summary = pd.DataFrame([
    {"rule": rule, "rows_count": int(mask.sum())}
    for rule, mask in checks.items()
])

display(business_rules_summary)

# Посмотрим несколько строк с критичными нарушениями.
critical_mask = checks["quantity <= 0"] | checks["unit_price <= 0"] | checks["discount < 0 or discount > 1"]
display(sales.loc[critical_mask, ["order_id", "order_date", "quantity", "unit_price", "discount", "product_id", "client_id", "region_id"]].head(15))

## 10. Проверка связности таблиц

    Проверим, все ли ключи из продаж есть в справочниках. Если ключ не найден, после объединения появятся пропуски, а отчёт по категориям, клиентам или регионам станет неполным.

In [ ]:
def clean_key(series):
    """Приводит ключ к строковому виду для проверки связности."""
    return series.astype("string").str.strip()

sales_product_key = clean_key(sales["product_id"])
products_product_key = clean_key(products["product_id"])

sales_client_key = clean_key(sales["client_id"])
clients_client_key = clean_key(clients["client_id"])

sales_region_key = clean_key(sales["region_id"])
regions_region_key = clean_key(regions["region_id"])

missing_product_mask = sales_product_key.notna() & ~sales_product_key.isin(set(products_product_key.dropna()))
missing_client_mask = sales_client_key.notna() & ~sales_client_key.isin(set(clients_client_key.dropna()))
missing_region_mask = sales_region_key.notna() & ~sales_region_key.isin(set(regions_region_key.dropna()))

missing_product_ids = sorted(sales_product_key[missing_product_mask].dropna().unique().tolist())
missing_client_ids = sorted(sales_client_key[missing_client_mask].dropna().unique().tolist())
missing_region_ids = sorted(sales_region_key[missing_region_mask].dropna().unique().tolist())

reference_summary = pd.DataFrame([
    {
        "reference_check": "sales.product_id -> products.product_id",
        "unknown_unique_keys": len(missing_product_ids),
        "affected_sales_rows": int(missing_product_mask.sum()),
        "unknown_keys": ", ".join(missing_product_ids),
    },
    {
        "reference_check": "sales.client_id -> clients.client_id",
        "unknown_unique_keys": len(missing_client_ids),
        "affected_sales_rows": int(missing_client_mask.sum()),
        "unknown_keys": ", ".join(missing_client_ids),
    },
    {
        "reference_check": "sales.region_id -> regions.region_id",
        "unknown_unique_keys": len(missing_region_ids),
        "affected_sales_rows": int(missing_region_mask.sum()),
        "unknown_keys": ", ".join(missing_region_ids),
    },
])

display(reference_summary)

## 11. Формирование таблицы проблем

    Соберём найденные проблемы в одну таблицу. Для каждой проблемы укажем набор данных, поле, тип проблемы, количество строк, критичность и предлагаемое действие.

In [ ]:
issue_rows = []

def add_issue(dataset, field, issue_type, rows_count, severity, suggested_action, details=""):
    rows_count = int(rows_count)
    if rows_count > 0:
        issue_rows.append({
            "dataset": dataset,
            "field": field,
            "issue_type": issue_type,
            "rows_count": rows_count,
            "severity": severity,
            "suggested_action": suggested_action,
            "details": details,
        })

add_issue("sales", "order_id", "duplicated_business_key", len(duplicated_orders), "critical", "deduplicate_or_confirm", "Повторяющиеся order_id могут завысить выручку")
add_issue("sales", "order_date", "invalid_date", sales["order_date_parsed"].isna().sum(), "high", "convert_or_request_source_check", "Строки с NaT нельзя использовать в динамике без проверки")
add_issue("sales", "quantity", "non_positive_quantity", (sales["quantity"] <= 0).sum(), "critical", "exclude_or_confirm", "Количество должно быть больше 0")
add_issue("sales", "unit_price", "non_positive_price", (sales["unit_price"] <= 0).sum(), "critical", "exclude_or_confirm", "Цена должна быть больше 0")
add_issue("sales", "discount", "discount_out_of_range", ((sales["discount"] < 0) | (sales["discount"] > 1)).sum(), "critical", "fix_or_confirm", "Скидка должна быть в диапазоне от 0 до 1")
add_issue("sales", "region_id", "missing_region_id", sales["region_id"].isna().sum(), "high", "restore_or_mark_unknown", "Без региона нельзя корректно сравнивать регионы")

add_issue("sales + products", "product_id", "missing_reference", missing_product_mask.sum(), "high", "check_products_dictionary", ", ".join(missing_product_ids))
add_issue("sales + clients", "client_id", "missing_reference", missing_client_mask.sum(), "medium", "check_clients_dictionary", ", ".join(missing_client_ids))
add_issue("sales + regions", "region_id", "missing_reference", missing_region_mask.sum(), "high", "check_regions_dictionary", ", ".join(missing_region_ids))

products_duplicate_key_rows = products[products_product_key.duplicated(keep=False)]
clients_duplicate_key_rows = clients[clients_client_key.duplicated(keep=False)]

add_issue("products", "product_id", "duplicated_reference_key", len(products_duplicate_key_rows), "high", "deduplicate_dictionary", "Дубли ключа в справочнике могут размножить строки после merge")
add_issue("clients", "client_id", "duplicated_reference_key", len(clients_duplicate_key_rows), "medium", "deduplicate_dictionary", "Дубли ключа в справочнике могут размножить строки после merge")

data_quality_issues = pd.DataFrame(issue_rows)
display(data_quality_issues)

## 12. Статус по каждому набору данных

    Теперь присвоим каждому набору данных статус: `ok`, `minor_issues`, `warning` или `needs_fix`.

In [ ]:
severity_rank = {"critical": 3, "high": 2, "medium": 1, "low": 0}

def define_status(dataset_name):
    related = data_quality_issues[
        data_quality_issues["dataset"].astype(str).str.contains(dataset_name, case=False, na=False)
    ]
    if related.empty:
        return "ok"
    max_rank = related["severity"].map(severity_rank).max()
    if max_rank >= 3:
        return "needs_fix"
    if max_rank == 2:
        return "warning"
    return "minor_issues"

data_quality_summary["status"] = data_quality_summary["dataset"].apply(define_status)

# Добавим количество проблем по датасету.
def count_related_issues(dataset_name):
    return int(data_quality_issues["dataset"].astype(str).str.contains(dataset_name, case=False, na=False).sum())

data_quality_summary["issues_count"] = data_quality_summary["dataset"].apply(count_related_issues)

display(data_quality_summary)

## 13. Сохранение результатов

    Сохраняем две таблицы и шаблон решения аналитика. После запуска ячейки откройте `outputs/analyst_decision.md` и замените шаблонные пункты на свои выводы по найденным проблемам.

In [ ]:
data_quality_summary.to_csv(output_dir / "data_quality_summary.csv", index=False)
data_quality_issues.to_csv(output_dir / "data_quality_issues.csv", index=False)

analyst_decision_template = """# Решение аналитика по входному аудиту данных

## Общий статус
Данные можно использовать для предварительного анализа только после исправления критичных ошибок.

## Критичные проблемы
1. Укажите критичную проблему 1.
2. Укажите критичную проблему 2.
3. Укажите критичную проблему 3.

## Что можно исправить автоматически
1. Укажите действие 1.
2. Укажите действие 2.

## Что нужно уточнить у владельца данных
1. Укажите вопрос 1.
2. Укажите вопрос 2.

## Ограничения будущего отчета
1. Укажите ограничение 1.
2. Укажите ограничение 2.

## Следующий шаг
Опишите, что нужно сделать перед расчетом итоговых показателей.
"""

(output_dir / "analyst_decision.md").write_text(analyst_decision_template, encoding="utf-8")

print("Файлы сохранены:")
for file_name in ["data_quality_summary.csv", "data_quality_issues.csv", "analyst_decision.md"]:
    print(output_dir / file_name)

## 14. Итоговая самопроверка

    Перед завершением проверьте:

    - все исходные файлы найдены;
    - notebook запускается сверху вниз;
    - даты преобразованы;
    - дубли `order_id` проверены;
    - бизнес-правила проверены;
    - связи со справочниками проверены;
    - файлы в `outputs` сохранены;
    - в `analyst_decision.md` есть конкретные проблемы, ограничения и следующий шаг.